# Qwen3ASRTextRMSNorm

In [1]:
class Qwen3ASRTextRMSNorm():
    pass

# Qwen3ASRTextAttention

In [ ]:
import torch
import torch.nn as nn

class Qwen3ASRTextAttention(nn.Module):
    
    # We leave the implementation of Qwen3ASRTextRMSNorm, pos_emb, mask, dropout, and kv Cache in the futrue

    def __init__(self,hidden_size,num_q_head,num_kv_head):

        self.hidden_size=hidden_size
        self.num_q_head=num_q_head
        self.num_kv_head=num_kv_head
        self.head_dim=hidden_size//num_q_head
        self.kv_group=num_q_head//num_kv_head
        self.scaling=self.head_dim**-0.5

        self.q=nn.Linear(hidden_size,self.head_dim*num_q_head)
        self.k=nn.Linear(hidden_size,self.head_dim*num_kv_head)
        self.v=nn.Linear(hidden_size,self.head_dim*num_kv_head)
        self.o=nn.Linear(hidden_size,hidden_size)
		
    def forward(self,x):
        
    	# x [B,T,F]
        B,T,_=x.shape()
        
    	# forward QKV
        # TODO why T should be at the dim=2 rather than dim=1 ?
        Q=self.q(x).view(B,self.num_q_head,T,self.head_dim)
        K=self.k(x).view(B,self.num_kv_head,T,self.head_dim)
        V=self.v(x).view(B,self.num_kv_head,T,self.head_dim)
        
        # repeat K and V
        K=torch.repeat_interleave(K,dim=1,repeats=self.kv_group)
        V=torch.repeat_interleave(V,dim=1,repeats=self.kv_group)
        
        # comute attention scores
        attn_weights=Q@K.transpose(-2,-1)*self.scaling
        attn_weights=nn.functional.softmax(attn_weights,dim=-1)
        output=(attn_weights@V).view(B,T,-1)
        output=self.o(output)

        return output, attn_weights